# Pipeline complet Deribit — BTC options (V2 · Plotly + ProviderFlow)

Même descente de pile que `demo_pipeline_deribit.ipynb`, mais avec des **graphes Plotly** et une
étape supplémentaire sur le **seam multi-provider** (`run_provider_flow`) : plusieurs providers
capturent dans **une seule** couche raw via le collecteur unifié (ADR 0027).

```
Connexion → Discovery → Capture (BrokerTick) → build_surface → Surface SVI → Risk → ProviderFlow
```

Discipline (`notebooks/README.md`) : le notebook ne fait qu'**appeler et tracer** les use-cases
d'orchestration testés (`build_surface`, `run_incremental_analytics`, `run_provider_flow`).
Aucune formule ni seuil métier ici — seulement le glue d'I/O Deribit (REST → `BrokerTick`) et
le rendu Plotly.

**Source : testnet `test.deribit.com`** (public, sans auth). Run :
`uv run --group notebooks jupyter lab notebooks/demo_pipeline_deribit_v2.ipynb`

In [ ]:
# %% Chart config — define once, apply everywhere
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# ── Palette ──────────────────────────────────────────────────────────────────
C = {
    "blue":      "#2563EB",  # primary series / calls
    "teal":      "#0D9488",  # puts / complementary
    "violet":    "#7C3AED",  # gamma / vega / 3rd series
    "amber":     "#D97706",  # warning / stress
    "red":       "#DC2626",  # rejected / calendar violation
    "green":     "#16A34A",  # usable / pass
    "indigo":    "#4F46E5",  # 2nd expiry series
    "slate900":  "#0F172A",  # titles
    "slate600":  "#475569",  # axis labels
    "slate400":  "#94A3B8",  # gridlines / zero lines
    "slate100":  "#F1F5F9",  # panel backgrounds
    "white":     "#FFFFFF",
}

DISCRETE = [C["blue"], C["teal"], C["violet"], C["amber"], C["indigo"], "#0EA5E9", "#F59E0B"]
FONT     = "Inter, IBM Plex Sans, -apple-system, sans-serif"

# ── Surface 3D params ─────────────────────────────────────────────────────────
SURFACE_COLORSCALE = [
    [0.00, "#1E3A5F"],  # deep navy   — low vol
    [0.25, "#2563EB"],  # blue
    [0.50, "#0D9488"],  # teal        — mid vol
    [0.75, "#D97706"],  # amber
    [1.00, "#DC2626"],  # red         — high vol / skew
]
SURFACE_CAMERA = dict(eye=dict(x=1.6, y=-1.6, z=0.9), up=dict(x=0, y=0, z=1))
SURFACE_ASPECT = dict(x=1.5, y=1.2, z=0.7)  # flatten z — IV range narrow vs k/t footprint

# ── Plotly template ───────────────────────────────────────────────────────────
_axis = dict(
    showgrid=True, gridcolor=C["slate400"], gridwidth=0.5,
    zeroline=False, linecolor=C["slate400"], linewidth=1,
    ticks="outside", tickcolor=C["slate400"],
    tickfont=dict(size=11), title_font=dict(size=12, color=C["slate600"]),
)
_tmpl = go.layout.Template()
_tmpl.layout = go.Layout(
    font=dict(family=FONT, size=12, color=C["slate600"]),
    title_font=dict(family=FONT, size=15, color=C["slate900"]),
    paper_bgcolor=C["white"],
    plot_bgcolor=C["white"],
    colorway=DISCRETE,
    xaxis=_axis, yaxis=_axis,
    legend=dict(
        bgcolor="rgba(255,255,255,0.85)", bordercolor=C["slate400"], borderwidth=1,
        font=dict(size=11), tracegroupgap=4,
    ),
    margin=dict(l=64, r=24, t=56, b=56),
    hoverlabel=dict(
        bgcolor=C["slate900"], font=dict(color=C["white"], size=12, family=FONT),
        bordercolor=C["slate900"],
    ),
    hovermode="closest",
)
pio.templates["algotrading"] = _tmpl
pio.templates.default = "plotly_white+algotrading"

print("Chart config loaded — template: algotrading (Plotly V2)")

In [ ]:

# %% Setup — imports, paramètres
from __future__ import annotations

from collections import defaultdict
from datetime import UTC, date, datetime
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from algotrading.core.config import (
    PlatformConfig,
    QcThresholdConfig,
    ScenarioConfig,
    SolverConfig,
    UniverseConfig,
    config_hash,
)
from algotrading.infra.collectors import BrokerTick, next_sequence
from algotrading.infra.connectivity import ManualClock
from algotrading.infra.contracts import InstrumentKey, InstrumentMaster, Position
from algotrading.infra.orchestration import (
    ProviderCapture,
    SurfaceJobRequest,
    build_surface,
    run_incremental_analytics,
    run_provider_flow,
)
from algotrading.infra.storage import ParquetStore
from algotrading.infra.surfaces import SviParams
from algotrading.infra_deribit.collectors.deribit_discovery import discover_instruments
from algotrading.infra_deribit.connectivity.deribit_transport import DeribitTransport

CURRENCY = "BTC"
MIN_DTE, MAX_DTE = 7, 45
STRIKE_BAND = 0.30
REST_BASE = "https://test.deribit.com/api/v2"
WS_BASE = "wss://test.deribit.com/ws/api/v2"

print(f"Cible : {CURRENCY} options, {MIN_DTE}–{MAX_DTE} DTE, ±{STRIKE_BAND:.0%} | {REST_BASE}")

## 1 · Connexion Deribit & prix index

In [ ]:

# %% Connexion + prix index
transport = DeribitTransport(rest_base=REST_BASE, ws_base=WS_BASE)
INDEX_PRICE = float(
    transport.get("/public/get_index_price", {"index_name": f"{CURRENCY.lower()}_usd"})["index_price"]
)
print(f"✓ Testnet Deribit — {CURRENCY}/USD index : ${INDEX_PRICE:,.0f}")

## 2 · Découverte → InstrumentMaster

`discover_instruments` parse les noms Deribit en `OptionContract` canoniques ; on en dérive les
`InstrumentMaster` (clé `InstrumentKey` + identité économique) que l'acteur consommera.

In [ ]:

# %% Découverte + masters
def _option_key(c) -> InstrumentKey:
    return InstrumentKey(
        underlying_symbol=c.symbol, security_type="OPT", exchange="DERIBIT", currency="USD",
        multiplier=float(c.multiplier),
        broker_contract_id=f"{c.symbol}-{c.expiry}-{c.strike}-{c.right.value}",
        expiry=c.expiry, strike=float(c.strike), option_right=c.right.value,
    )


underlying_key = InstrumentKey(
    underlying_symbol=CURRENCY, security_type="CRYPTO", exchange="DERIBIT", currency="USD",
    multiplier=1.0, broker_contract_id=CURRENCY,
)

AS_OF = datetime.now(UTC)
all_contracts = discover_instruments(transport, CURRENCY, min_days=MIN_DTE, max_days=MAX_DTE)
lo, hi = (1 - STRIKE_BAND) * INDEX_PRICE, (1 + STRIKE_BAND) * INDEX_PRICE
contracts = [c for c in all_contracts if lo <= float(c.strike) <= hi]

masters: list[InstrumentMaster] = [
    InstrumentMaster(instrument_key=underlying_key.canonical(), as_of_date=AS_OF.date(),
                     instrument=underlying_key, raw_broker_payload="{}")
]
contract_by_key: dict[str, object] = {}
for c in contracts:
    key = _option_key(c)
    contract_by_key[key.canonical()] = c
    masters.append(InstrumentMaster(instrument_key=key.canonical(), as_of_date=AS_OF.date(),
                                    instrument=key, raw_broker_payload="{}"))

by_expiry: dict[date, int] = defaultdict(int)
for c in contracts:
    by_expiry[c.expiry] += 1
expiries_sorted = sorted(by_expiry)
print(f"Découverts {len(all_contracts)} → retenus {len(contracts)} ({len(expiries_sorted)} expiries)")

In [ ]:

# %% Plot 1 — Contrats par DTE
dtes = [(e - AS_OF.date()).days for e in expiries_sorted]
counts = [by_expiry[e] for e in expiries_sorted]
fig = go.Figure(go.Bar(
    x=dtes, y=counts, marker_color=C["blue"], marker_line_color=C["white"], marker_line_width=0.5,
    opacity=0.9, customdata=[str(e) for e in expiries_sorted],
    hovertemplate="<b>%{customdata}</b><br>DTE: %{x}d<br>Contrats: %{y:,}<extra></extra>",
))
fig.update_layout(title=f"{CURRENCY} — Open Contracts by Days to Expiry",
                  xaxis_title="DTE (days)", yaxis_title="Contract count", bargap=0.25, height=350)
fig.show()

## 3 · Capture REST → `BrokerTick` → `build_surface`

Un ticker REST par contrat → `BrokerTick` (shape EAV push, ADR 0027), rejoués par un adaptateur
push que `build_surface` *pompe*. Glue d'I/O uniquement.

In [ ]:

# %% Adaptateur push REST + fetch tickers
class RestSnapshotAdapter:
    """MarketDataAdapter push minimal : rejoue une liste figée de BrokerTick quand pompé."""

    def __init__(self, ticks: list[BrokerTick]) -> None:
        self._ticks = ticks
        self._cb = None

    def subscribe(self, instrument_keys: object) -> None: ...
    def set_tick_callback(self, callback) -> None:  # noqa: ANN001
        self._cb = callback
    def set_fault_callback(self, callback) -> None: ...  # noqa: ANN001
    def unsubscribe_all(self) -> None: ...

    def pump(self, _collector: object) -> None:
        for tick in self._ticks:
            self._cb(tick)  # type: ignore[misc]


def _deribit_name(c) -> str:
    return f"{c.symbol}-{c.expiry.day}{c.expiry.strftime('%b%y').upper()}-{int(c.strike)}-{c.right.value}"


_DERIBIT_FIELDS = {"best_bid_price": "bid", "best_ask_price": "ask",
                   "last_price": "last", "mark_price": "mark_price"}
_counters: dict[tuple[str, str], int] = {}
ticks: list[BrokerTick] = []


def _emit(key: str, field: str, value: float) -> None:
    ticks.append(BrokerTick(instrument_key=key, field_name=field, value=value, underlying=CURRENCY,
                            sequence=next_sequence(_counters, key, field), exchange_ts=AS_OF))


half = INDEX_PRICE * 0.0001
_emit(underlying_key.canonical(), "bid", INDEX_PRICE - half)
_emit(underlying_key.canonical(), "ask", INDEX_PRICE + half)
_emit(underlying_key.canonical(), "last", INDEX_PRICE)

n_fetched = 0
for canonical, c in contract_by_key.items():
    try:
        ticker = transport.get("/public/ticker", {"instrument_name": _deribit_name(c)})
    except Exception:  # noqa: BLE001 — un contrat fautif n'avorte pas la boucle
        continue
    for src, canon in _DERIBIT_FIELDS.items():
        raw = ticker.get(src)
        if raw is not None and float(raw) > 0:
            _emit(canonical, canon, round(float(raw) * INDEX_PRICE, 6))
    n_fetched += 1

transport.close()
print(f"Tickers lus : {n_fetched}  |  BrokerTicks : {len(ticks)}")

In [ ]:

# %% build_surface — capture + acteur + surface SVI
config = PlatformConfig(
    universe=UniverseConfig(version="deribit-btc-nb", underlyings=(CURRENCY,), exchange="DERIBIT"),
    qc_threshold=QcThresholdConfig(version="deribit-btc-nb", max_spread_pct=0.60,
                                   max_quote_age_seconds=600.0, min_chain_count=2),
    solver=SolverConfig(version="deribit-btc-nb", iv_tolerance=1e-8, max_iterations=100),
    scenario=ScenarioConfig(version="deribit-btc-nb", spot_shocks=(-0.20, -0.10, 0.0, 0.10, 0.20),
                            vol_shocks=(-0.10, 0.0, 0.10)),
)
CONFIG_HASH = config_hash(config)

adapter = RestSnapshotAdapter(ticks)
_tmp = TemporaryDirectory(prefix="deribit-surface-v2-")
store = ParquetStore(Path(_tmp.name))

surface = build_surface(
    request=SurfaceJobRequest(symbol=CURRENCY, trade_date=AS_OF.date(), market_data_type=3,
                              as_of=AS_OF, calc_ts=AS_OF, persist=True),
    store=store, config=config, config_hash=CONFIG_HASH,
    adapter=adapter, masters=masters, drive=adapter.pump,
    clock=ManualClock(start=AS_OF), correlation_id="notebook-deribit-v2",
)
outputs = surface.outputs
print(f"Events {surface.collection.event_count} | maturités fittées {surface.fitted_maturities} "
      f"| config_hash {CONFIG_HASH[:12]}…")
print(f"Snapshots {len(outputs.snapshots)} | IvPoints {len(outputs.iv_points)} "
      f"| SVI params {len(outputs.surface_parameters)}")

## 4 · Surface SVI — fit par expiry, smile, surface 3D

In [ ]:

# %% Helpers de lecture des sorties acteur
params_by_expiry = {p.expiry_date: p for p in outputs.surface_parameters}


def _expiry_of(contract_key: str) -> date | None:
    field = contract_key.split("|")[6]  # slot expiry de InstrumentKey.canonical()
    return date.fromisoformat(field) if field else None


def _svi(p) -> SviParams:
    return SviParams(a=p.svi_a, b=p.svi_b, rho=p.svi_rho, m=p.svi_m, sigma=p.svi_sigma)


market_by_expiry: dict[date, list[tuple[float, float]]] = defaultdict(list)
smile_by_expiry: dict[date, list] = defaultdict(list)
for iv in outputs.iv_points:
    exp = _expiry_of(iv.contract_key)
    if exp not in params_by_expiry:
        continue
    if iv.k is not None and iv.total_variance is not None:
        market_by_expiry[exp].append((iv.k, iv.total_variance))
    if iv.k is not None and iv.iv is not None:
        smile_by_expiry[exp].append(iv)

expiries_fit = sorted(params_by_expiry)
print(f"{'Expiry':>12} {'DTE':>4} {'n':>3} {'ATM vol':>8} {'RMSE':>10} {'arb-free':>9}")
for s in sorted(surface.slices, key=lambda s: s.maturity_years):
    dte = int(s.maturity_years * 365)
    atm = f"{s.atm_vol * 100:.1f}%" if s.atm_vol is not None else "n/a"
    rmse = f"{s.rmse:.5f}" if s.rmse is not None else "n/a"
    print(f"{str(s.expiry_date):>12} {dte:>4} {s.n_points:>3} {atm:>8} {rmse:>10} {str(s.arb_free):>9}")

In [ ]:

# %% Plot 2 — SVI fit vs marché, par expiry
if not expiries_fit:
    print("Aucune slice — plot ignoré.")
else:
    fig = make_subplots(rows=1, cols=len(expiries_fit),
                        subplot_titles=[f"{e} ({(e - AS_OF.date()).days}j)" for e in expiries_fit])
    for col, exp in enumerate(expiries_fit, start=1):
        svi = _svi(params_by_expiry[exp])
        pts = market_by_expiry.get(exp, [])
        if pts:
            ks = [k for k, _ in pts]
            fig.add_trace(go.Scatter(x=ks, y=[v for _, v in pts], mode="markers",
                                     marker=dict(color=C["amber"], size=7), name="Marché",
                                     showlegend=col == 1), row=1, col=col)
            grid = np.linspace(min(ks) - 0.05, max(ks) + 0.05, 200)
        else:
            grid = np.linspace(-0.4, 0.4, 200)
        fig.add_trace(go.Scatter(x=grid, y=[svi.total_variance(float(k)) for k in grid], mode="lines",
                                 line=dict(color=C["blue"], width=2.5), name="SVI",
                                 showlegend=col == 1), row=1, col=col)
        fig.update_xaxes(title_text="ln(K/F)", row=1, col=col)
    fig.update_yaxes(title_text="σ²T", row=1, col=1)
    fig.update_layout(title=f"{CURRENCY} — SVI calibration par expiry", height=380)
    fig.show()

In [ ]:

# %% Plot 3 — Smile IV par expiry
if not smile_by_expiry:
    print("Aucun IvPoint résolu — plot ignoré.")
else:
    exps = sorted(smile_by_expiry)
    fig = go.Figure()
    for i, exp in enumerate(exps):
        pts = sorted(smile_by_expiry[exp], key=lambda iv: iv.k)
        fig.add_trace(go.Scatter(
            x=[iv.k for iv in pts], y=[iv.iv * 100 for iv in pts], mode="lines+markers",
            line=dict(color=DISCRETE[i % len(DISCRETE)], width=2),
            name=f"{exp} ({(exp - AS_OF.date()).days}j)",
        ))
    fig.add_vline(x=0, line=dict(color=C["slate400"], width=1, dash="dash"))
    fig.update_layout(title=f"{CURRENCY} — Smile de volatilité implicite",
                      xaxis_title="Log-moneyness ln(K/F)", yaxis_title="IV (%)", height=420)
    fig.show()

In [ ]:

# %% Plot 4 — Surface 3D (variance totale échantillonnée sur les SVI fittées)
if len(expiries_fit) < 2:
    print("Moins de 2 maturités fittées — surface 3D ignorée.")
else:
    k_axis = np.linspace(-0.3, 0.3, 40)
    mats = [params_by_expiry[e].maturity_years for e in expiries_fit]
    Z = np.array([[_svi(params_by_expiry[e]).total_variance(float(k)) for k in k_axis]
                  for e in expiries_fit])
    # variance totale → IV annualisée
    iv_grid = np.sqrt(np.maximum(Z, 0) / np.array(mats)[:, None]) * 100
    fig = go.Figure(go.Surface(
        x=k_axis, y=[m * 365 for m in mats], z=iv_grid,
        colorscale=SURFACE_COLORSCALE, colorbar=dict(title="IV (%)"),
    ))
    fig.update_layout(
        title=f"{CURRENCY} — Surface de volatilité SVI",
        scene=dict(xaxis_title="ln(K/F)", yaxis_title="DTE (jours)", zaxis_title="IV (%)",
                   camera=SURFACE_CAMERA, aspectratio=SURFACE_ASPECT),
        height=620,
    )
    fig.show()

## 5 · Risque & scénarios — straddle ATM

`run_incremental_analytics` valorise un straddle ATM (call + put même strike/expiry, sur une
expiry fittée) : Greeks nets + PnL par scénario du grid de la config.

In [ ]:

# %% Straddle ATM + analytics
fitted = set(params_by_expiry)
options = [m for m in masters if m.instrument.is_option() and m.instrument.expiry in fitted]
scenarios: list = []
risk = None

if not options:
    print("Aucune option sur une expiry fittée — risque ignoré.")
else:
    atm = min(options, key=lambda m: abs(m.instrument.strike - INDEX_PRICE))
    sk, ex = atm.instrument.strike, atm.instrument.expiry

    def _leg(right):  # noqa: ANN001, ANN202
        return next((m for m in options if m.instrument.strike == sk
                     and m.instrument.expiry == ex and m.instrument.option_right == right), None)

    call_leg, put_leg = _leg("C"), _leg("P")
    if call_leg is None or put_leg is None:
        print("Call/put ATM introuvable — risque ignoré.")
    else:
        positions = [
            Position(valuation_ts=AS_OF, portfolio_id="nb-straddle",
                     contract_key=call_leg.instrument.canonical(), quantity=1.0, source="notebook"),
            Position(valuation_ts=AS_OF, portfolio_id="nb-straddle",
                     contract_key=put_leg.instrument.canonical(), quantity=1.0, source="notebook"),
        ]
        analytics = run_incremental_analytics(
            store=store, config=config, config_hash=CONFIG_HASH, positions=positions,
            instruments=[m.instrument for m in masters], masters=masters,
            trade_date=AS_OF.date(), as_of=AS_OF, calc_ts=AS_OF,
            clock=ManualClock(start=AS_OF), correlation_id="nb-deribit-v2-risk", persist=False,
        )
        scenarios = list(analytics.outputs.scenarios)
        aggs = analytics.outputs.risk_aggregates
        risk = aggs[0] if aggs else None
        print(f"Straddle ATM K={float(sk):,.0f} {ex}")
        if risk is not None:
            print(f"  Δ {risk.net_delta:+.4f}  Γ {risk.net_gamma:+.8f}  "
                  f"Vega {risk.net_vega:+.2f}  Θ {risk.net_theta:+.2f}")

In [ ]:

# %% Plot 5 — PnL par scénario
if not scenarios:
    print("Pas de scénarios — plot ignoré.")
else:
    pnl: dict[str, float] = defaultdict(float)
    for s in scenarios:
        pnl[s.scenario_id] += s.pnl
    names = list(pnl)
    vals = [pnl[n] for n in names]
    fig = go.Figure(go.Bar(
        x=vals, y=names, orientation="h",
        marker_color=[C["green"] if v >= 0 else C["red"] for v in vals],
        hovertemplate="%{y}<br>PnL: $%{x:,.0f}<extra></extra>",
    ))
    fig.add_vline(x=0, line=dict(color=C["slate900"], width=1))
    fig.update_layout(title=f"{CURRENCY} straddle ATM — PnL par scénario (full reprice)",
                      xaxis_title="PnL (USD)", height=420, yaxis=dict(autorange="reversed"))
    fig.show()

## 6 · Seam multi-provider — `run_provider_flow`

Le seam post-dimension-provider : plusieurs `ProviderCapture` capturent dans **une seule** couche
raw partagée via le collecteur unifié (ADR 0027). On splitte ici les ticks Deribit en deux
providers fictifs (`DERIBIT-A` / `DERIBIT-B`) pour démontrer l'attribution par session — chaque
provider écrit ses events, l'union reconstitue tout, et chaque session reste traçable.

In [ ]:

# %% run_provider_flow — deux providers, une couche raw
half = len(ticks) // 2
flow_tmp = TemporaryDirectory(prefix="deribit-flow-v2-")
flow_store = ParquetStore(Path(flow_tmp.name))

adapter_a = RestSnapshotAdapter(ticks[:half])
adapter_b = RestSnapshotAdapter(ticks[half:])
subscribe = sorted({t.instrument_key for t in ticks})

flow = run_provider_flow(
    store=flow_store,
    providers=[
        ProviderCapture(provider="DERIBIT-A", adapter=adapter_a, subscribe=subscribe, drive=adapter_a.pump),
        ProviderCapture(provider="DERIBIT-B", adapter=adapter_b, subscribe=subscribe, drive=adapter_b.pump),
    ],
    trade_date=AS_OF.date(), clock=ManualClock(start=AS_OF), correlation_id="nb-deribit-v2-flow",
)

raw_all = flow_store.read("raw_market_events")
sessions = sorted({e.session_id for e in raw_all})
print(f"Providers : {len(flow.captures)}  |  events totaux (union) : {len(raw_all)}")
print(f"Sessions traçables : {sessions}")
per_session: dict[str, int] = defaultdict(int)
for e in raw_all:
    per_session[e.session_id] += 1
for s in sessions:
    print(f"  {s} : {per_session[s]} events")

_tmp.cleanup()
flow_tmp.cleanup()

## Récapitulatif de la pile

| Étape | Module | Entrée → Sortie |
|-------|--------|-----------------|
| 1 Connexion | `infra_deribit.connectivity` | REST testnet → `index_price` |
| 2 Discovery | `infra_deribit.collectors.deribit_discovery` | REST → `OptionContract[]` → `InstrumentMaster[]` |
| 3 Capture | `infra.collectors` (`BrokerTick`) | tickers REST → `BrokerTick[]` (seam push, ADR 0027) |
| 4 Surface | `infra.orchestration.build_surface` | ticks + masters → acteur → `SurfaceJobResult` |
| 5 Risk + scénarios | `infra.orchestration.run_incremental_analytics` | positions → `RiskAggregate[]` + `ScenarioResult[]` |
| 6 ProviderFlow | `infra.orchestration.run_provider_flow` | N providers → **une** couche raw, attribution par session |

**Propriétés clés** : un seul driver (l'acteur) ; le notebook n'appelle que des use-cases testés,
zéro formule. Provenance (`config_hash`) sur chaque dérivé. Source publique testnet, sans auth.